# Feature Engineering — Ngày Lễ (Holiday)

| Feature | Mô tả |
|---|---|
| `is_national_holiday` | Ngày lễ quốc gia (tất cả stores) |
| `is_regional_holiday` | Ngày lễ vùng (khớp state) |
| `is_local_holiday` | Ngày lễ địa phương (khớp city) |
| `is_transferred_holiday` | Make-up day cho ngày lễ đã bị dời (type = Transfer) |
| `holiday_type` | Mã loại ngày lễ (0–5) |
| `is_carnaval_feature` | Flag Carnaval |
| `days_to_next_holiday` | Số ngày đến ngày lễ tiếp theo |
| `days_after_last_holiday` | Số ngày kể từ ngày lễ gần nhất |
| `is_earthquake_period` | Flag tháng sau động đất Ecuador 4/2016 |

In [1]:
import pandas as pd
import numpy as np

In [ ]:
import sys
from pathlib import Path

def _find_root():
    """Tìm project root chứa config.py."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.py').exists():
            return p
    raise RuntimeError("Không tìm thấy project root. Mở Jupyter từ thư mục gốc của project.")

_root = _find_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROCESSED_DIR

df_train    = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_holidays = pd.read_csv(PROCESSED_DIR / 'holidays_events_cleaned.csv')
df_stores   = pd.read_csv(PROCESSED_DIR / 'stores_cleaned.csv')

print(f'Train shape    : {df_train.shape}')
print(f'Holidays shape : {df_holidays.shape}')
print(f'Stores shape   : {df_stores.shape}')

---
## 1. Parse Dates

In [3]:
df_train['date']    = pd.to_datetime(df_train['date'])
df_holidays['date'] = pd.to_datetime(df_holidays['date'])

print('Date columns converted.')
print(f"Train date range   : {df_train['date'].min()} → {df_train['date'].max()}")
print(f"Holiday date range : {df_holidays['date'].min()} → {df_holidays['date'].max()}")

Date columns converted.
Train date range   : 2013-01-01 00:00:00 → 2017-08-15 00:00:00
Holiday date range : 2012-03-02 00:00:00 → 2017-12-26 00:00:00


---
## 2. Lọc Transferred Holidays

Chỉ giữ `transferred == False` (ngày lễ thực sự xảy ra).  
`is_transferred_holiday` đánh dấu make-up days có `type == 'Transfer'`.

In [ ]:
valid_holidays = df_holidays[df_holidays['transferred'] == False].copy()

# is_transferred_holiday = 1 nếu đây là ngày "make-up" cho một ngày lễ đã bị dời (type='Transfer').
# KHÔNG dùng cột 'transferred' — sau khi filter transferred==False, cột đó luôn = 0.
valid_holidays['is_transferred'] = (valid_holidays['type'] == 'Transfer').astype(int)

print(f'Total holiday rows       : {len(df_holidays)}')
print(f'Valid (non-transferred)  : {len(valid_holidays)}')
print(f'Removed (transferred)    : {len(df_holidays) - len(valid_holidays)}')
print(f'Transfer-type (make-up)  : {valid_holidays["is_transferred"].sum()}')
valid_holidays[['date', 'type', 'locale', 'locale_name', 'transferred']].head()

---
## 3. Encode Holiday Type

`Work Day`=0, `Holiday`=1, `Event`=2, `Additional`=3, `Transfer`=4, `Bridge`=5

In [5]:
type_mapping = {
    'Holiday'   : 1,
    'Event'     : 2,
    'Additional': 3,
    'Transfer'  : 4,
    'Bridge'    : 5,
    'Work Day'  : 0
}

valid_holidays['holiday_type_encoded'] = valid_holidays['type'].map(type_mapping).fillna(0)

print('Holiday type distribution after encoding:')
valid_holidays.groupby(['type', 'holiday_type_encoded']).size().reset_index(name='count')

Holiday type distribution after encoding:


,type,holiday_type_encoded,count
0,Additional,3,51
1,Bridge,5,5
2,Event,2,56
3,Holiday,1,209
4,Transfer,4,12
5,Work Day,0,5


---
## 4. Flag Carnaval

In [6]:
valid_holidays['is_carnaval'] = (
    valid_holidays['description']
    .str.contains('Carnaval', case=False, na=False)
    .astype(int)
)

carnaval_dates = valid_holidays[valid_holidays['is_carnaval'] == 1][['date', 'description', 'locale']]
print(f'Carnaval event rows found: {len(carnaval_dates)}')
carnaval_dates

Carnaval event rows found: 10


,date,description,locale
44,2013-02-11,Carnaval,National
45,2013-02-12,Carnaval,National
94,2014-03-03,Carnaval,National
95,2014-03-04,Carnaval,National
162,2015-02-16,Carnaval,National
163,2015-02-17,Carnaval,National
212,2016-02-08,Carnaval,National
213,2016-02-09,Carnaval,National
299,2017-02-27,Carnaval,National
300,2017-02-28,Carnaval,National


---
## 5. Merge Phạm vi Địa lý (National / Regional / Local)

In [7]:
df = df_train.merge(
    df_stores[['store_nbr', 'city', 'state']],
    on='store_nbr',
    how='left'
)

print(f'Merged dataframe shape : {df.shape}')
print(f'Unique stores          : {df["store_nbr"].nunique()}')
print(f'States in data         : {sorted(df["state"].unique())}')
df[['date', 'store_nbr', 'city', 'state']].drop_duplicates('store_nbr').head(5)

Merged dataframe shape : (3000888, 8)
Unique stores          : 54
States in data         : ['Azuay', 'Bolivar', 'Chimborazo', 'Cotopaxi', 'El Oro', 'Esmeraldas', 'Guayas', 'Imbabura', 'Loja', 'Los Rios', 'Manabi', 'Pastaza', 'Pichincha', 'Santa Elena', 'Santo Domingo de los Tsachilas', 'Tungurahua']


,date,store_nbr,city,state
0,2013-01-01,1,Quito,Pichincha
33,2013-01-01,10,Quito,Pichincha
66,2013-01-01,11,Cayambe,Pichincha
99,2013-01-01,12,Latacunga,Cotopaxi
132,2013-01-01,13,Latacunga,Cotopaxi


---
## 6. Áp dụng Holiday Flags

In [8]:
holiday_features = []

for _, row in valid_holidays.iterrows():
    h_date       = row['date']
    h_locale     = row['locale']
    h_locale_name = row['locale_name']
    h_encoded    = row['holiday_type_encoded']
    is_transferred = row['is_transferred']
    is_carnaval  = row['is_carnaval']

    if h_locale == 'National':
        mask = (df['date'] == h_date)
    elif h_locale == 'Regional':
        mask = (df['date'] == h_date) & (df['state'] == h_locale_name)
    elif h_locale == 'Local':
        mask = (df['date'] == h_date) & (df['city'] == h_locale_name)
    else:
        continue

    holiday_features.append({
        'mask'         : mask,
        'locale'       : h_locale,
        'is_transferred': is_transferred,
        'encoded_type' : h_encoded,
        'is_carnaval'  : is_carnaval
    })

print(f'Total holiday mask entries built: {len(holiday_features)}')

Total holiday mask entries built: 338


In [9]:
df['is_national_holiday']  = 0
df['is_regional_holiday']  = 0
df['is_local_holiday']     = 0
df['is_transferred_holiday'] = 0
df['holiday_type']         = 0
df['is_carnaval_feature']  = 0

print('Holiday columns initialised to 0:')
print(df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
          'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature']].sum())

Holiday columns initialised to 0:
is_national_holiday       0
is_regional_holiday       0
is_local_holiday          0
is_transferred_holiday    0
holiday_type              0
is_carnaval_feature       0
dtype: int64


In [10]:
for h in holiday_features:
    if h['locale'] == 'National':
        df.loc[h['mask'], 'is_national_holiday'] = 1
    elif h['locale'] == 'Regional':
        df.loc[h['mask'], 'is_regional_holiday'] = 1
    elif h['locale'] == 'Local':
        df.loc[h['mask'], 'is_local_holiday'] = 1

    df.loc[h['mask'], 'is_transferred_holiday'] = h['is_transferred']
    df.loc[h['mask'], 'holiday_type']           = h['encoded_type']
    df.loc[h['mask'], 'is_carnaval_feature']    = h['is_carnaval']

print('Holiday flags assigned. Summary:')
print(df[['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
          'is_transferred_holiday', 'is_carnaval_feature']].sum())

Holiday flags assigned. Summary:
is_national_holiday       242352
is_regional_holiday         1023
is_local_holiday           11880
is_transferred_holiday         0
is_carnaval_feature        17820
dtype: int64


---
## 7. Halo Effect (days_to_next / days_after_last)

In [11]:
unique_holiday_dates = (
    valid_holidays['date']
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

def get_days_to_next(date):
    """Return the number of days until the next holiday, or -1 if none exists."""
    future = unique_holiday_dates[unique_holiday_dates > date]
    return (future.iloc[0] - date).days if not future.empty else -1

def get_days_after_last(date):
    """Return the number of days since the last holiday, or -1 if none exists."""
    past = unique_holiday_dates[unique_holiday_dates < date]
    return (date - past.iloc[-1]).days if not past.empty else -1

# Compute on unique dates only (much faster)
unique_dates = df[['date']].drop_duplicates().copy()
unique_dates['days_to_next_holiday']   = unique_dates['date'].apply(get_days_to_next)
unique_dates['days_after_last_holiday'] = unique_dates['date'].apply(get_days_after_last)

# Merge back to the main dataframe
df = df.merge(unique_dates, on='date', how='left')

print('Halo effect features added:')
print(df[['days_to_next_holiday', 'days_after_last_holiday']].describe())

Halo effect features added:
       days_to_next_holiday  days_after_last_holiday
count          3.000888e+06             3.000888e+06
mean           1.072862e+01             1.072387e+01
std            1.052096e+01             1.052355e+01
min            1.000000e+00             1.000000e+00
25%            3.000000e+00             3.000000e+00
50%            7.000000e+00             7.000000e+00
75%            1.600000e+01             1.600000e+01
max            6.000000e+01             6.000000e+01


---
## 8. Earthquake Dummy (2016-04-16 → 2016-05-16)

In [12]:
earthquake_start = pd.to_datetime('2016-04-16')
earthquake_end   = pd.to_datetime('2016-05-16')

df['is_earthquake_period'] = (
    (df['date'] >= earthquake_start) & (df['date'] <= earthquake_end)
).astype(int)

n_earthquake_rows = df['is_earthquake_period'].sum()
print(f'Rows flagged as earthquake period : {n_earthquake_rows}')
print(f'Unique dates in earthquake period  : {df[df["is_earthquake_period"]==1]["date"].nunique()}')

Rows flagged as earthquake period : 55242
Unique dates in earthquake period  : 31


---
## 9. Drop Temporary Columns

In [13]:
df = df.drop(columns=['city', 'state'])

print(f'Final dataframe shape : {df.shape}')
print('\nColumns:')
print(df.columns.tolist())

Final dataframe shape : (3000888, 15)

Columns:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature', 'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period']


---
## 10. Output Preview

In [14]:
holiday_cols = [
    'is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
    'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature',
    'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period'
]

print('=== Feature Summary ===')
print(df[holiday_cols].describe().T[['mean', 'min', 'max']])
print()
print('=== Sample rows with active holidays ===')
df[df['is_national_holiday'] == 1][['date', 'store_nbr'] + holiday_cols].head(10)

=== Feature Summary ===
                              mean  min   max
is_national_holiday       0.080760  0.0   1.0
is_regional_holiday       0.000341  0.0   1.0
is_local_holiday          0.003959  0.0   1.0
is_transferred_holiday    0.000000  0.0   0.0
holiday_type              0.167645  0.0   5.0
is_carnaval_feature       0.005938  0.0   1.0
days_to_next_holiday     10.728622  1.0  60.0
days_after_last_holiday  10.723872  1.0  60.0
is_earthquake_period      0.018409  0.0   1.0

=== Sample rows with active holidays ===


,date,store_nbr,is_national_holiday,is_regional_holiday,is_local_holiday,is_transferred_holiday,holiday_type,is_carnaval_feature,days_to_next_holiday,days_after_last_holiday,is_earthquake_period
0,2013-01-01,1,1,0,0,0,1,0,4,1,0
1,2013-01-01,1,1,0,0,0,1,0,4,1,0
2,2013-01-01,1,1,0,0,0,1,0,4,1,0
3,2013-01-01,1,1,0,0,0,1,0,4,1,0
4,2013-01-01,1,1,0,0,0,1,0,4,1,0
5,2013-01-01,1,1,0,0,0,1,0,4,1,0
6,2013-01-01,1,1,0,0,0,1,0,4,1,0
7,2013-01-01,1,1,0,0,0,1,0,4,1,0
8,2013-01-01,1,1,0,0,0,1,0,4,1,0
9,2013-01-01,1,1,0,0,0,1,0,4,1,0


In [ ]:
# CSV write disabled — tất cả features được merge trong build_train_final.py.
# Để tạo CSV cho model training:
#   python scripts/build_train_final.py   →  data/processed/train_final.csv
#   python scripts/create_splits.py       →  data/processed/splits/
print('[SKIPPED] CSV write disabled')
print(f'Feature dataframe shape : {df.shape}')